# Game Discovery Analytics — ETL Pipeline

**Author:** Don John
**Type:** End-to-end data engineering project (Extract → Transform → Load → Analytics)

## Overview

This project implements a full ETL pipeline that pulls game catalog data from the public **RAWG Video Games Database API**, cleans and enriches it with Python/pandas, and loads it into **Google BigQuery** using a lightweight star-schema design so it can be queried directly from a BI tool such as Looker Studio.

The pipeline was built to answer product/marketing-style analytics questions that a game discovery platform would realistically care about, for example:

- Which genres are released most often, and which ones rate the highest?
- Which platforms dominate the market by title count?
- Is there a correlation between critic scores (Metacritic) and player ratings?
- Which developers consistently ship highly-rated games?
- How have release trends shifted over the last several years?

## Architecture

```
RAWG.io API  (rawg.io/apidocs — free API key)
      │
      │  Extract:
      │  • /games       → core catalog data (rating, metacritic, playtime, etc.)
      │  • /genres      → genre reference data
      │  • /platforms   → platform reference data
      ▼
Python ETL (requests + pandas)
      │
      └──► BigQuery (OLAP) — dataset: RAWG
           ├── fact_games        (denormalized fact table)
           ├── genres_raw        (game ↔ genre junction table)
           ├── platforms_raw     (game ↔ platform junction table)
           ├── genre             (genre dimension)
           └── platform          (platform dimension)
                     │
                     ▼
              Looker Studio Dashboard
```

## Tech Stack

| Layer | Tools |
|---|---|
| Extraction | Python, `requests`, RAWG REST API |
| Transformation | `pandas`, `numpy` |
| Storage / Warehouse | Google BigQuery |
| Credentials & config | `python-dotenv`, Google service account |
| Intended consumption layer | Looker Studio (Custom SQL) |

## About the RAWG API

| Parameter | Detail |
|---|---|
| **Base URL** | `https://api.rawg.io/api/` |
| **Auth** | API key passed as a query parameter: `?key=YOUR_KEY` |
| **Getting a key** | Free registration at https://rawg.io/apidocs |
| **Rate limit** | ~20 requests/second — fairly generous |
| **Pagination** | Cursor-style pagination via the `next` field in each response |

---


## 1. Environment Setup

Load the required libraries and environment variables (API key, BigQuery project ID, and service account credential path) from a local `.env` file so no secrets are hard-coded in the notebook.

In [32]:
import pandas as pd
import numpy as np
import pandas_gbq
import requests

from google.oauth2 import service_account
from dotenv import load_dotenv
import os

load_dotenv()

True

---
## 2. Extraction Layer

### 2.1 Extracting `/games` with Pagination

The RAWG `/games` endpoint returns paginated results. The extraction function below:

1. Fetches multiple pages of results from `GET /games`.
2. Follows pagination through the `next` field returned in the API response.
3. Keeps nested fields (`genres`, `platforms`) intact for now — they get flattened separately in the transform step, since each game can belong to multiple genres/platforms.
4. Keeps unreleased titles (`tba=True` or `released=None`) in the dataset rather than dropping them, since "not yet released" is itself a useful signal for trend analysis.

Key fields captured per game: `id`, `name`, `released`, `tba`, `rating`, `ratings_count`, `metacritic`, `playtime`, `reviews_count`, `esrb_rating`, `genres`, `platforms`.


In [33]:
def get_data(endpoint:str, params=None, base_url="https://api.rawg.io/api"):
    if params is None:
        params = {}
    
    params["key"] = os.getenv("API_KEY")
    url_endpoint = base_url + "/" + endpoint
    response = requests.get(url=url_endpoint, params=params)
    
    if response.status_code == 200: 
        return response
    else:
        raise ValueError(response.status_code)

In [34]:
# Quick smoke test to confirm the API key and connection work before running a full pull
response = get_data(endpoint="games")
if response:
    print("Test Succeed")
else:
    print("Failed")


Test Succeed


In [35]:
def fetch_pages(endpoint:str, params=None, base_url="https://api.rawg.io/api", max_page=5):
    result = []
    
    if params is None:
        params = {}
    
    params["key"] = os.getenv("API_KEY")
    url_endpoint = base_url + "/" + endpoint
    
    for i in range(max_page):
        response = requests.get(url=url_endpoint, params=params)
        if response.status_code == 200: 
            response = response.json()
            result.extend(response["results"])
            if response["next"]:
                response = requests.get(url=response["next"], params=params).json()
            else:
                break
        else:
            raise ValueError(response.status_code)
    
    return result

In [36]:
query_params = {"page_size": 40}

fetch_result = fetch_pages(endpoint="games", params=query_params)

len(fetch_result)

200

In [37]:
df_raw = pd.DataFrame(fetch_result)
df_raw.head()

,id,slug,name,released,tba,background_image,rating,rating_top,ratings,ratings_count,...,saturated_color,dominant_color,platforms,parent_platforms,genres,stores,clip,tags,esrb_rating,short_screenshots
0,3498,grand-theft-auto-v,Grand Theft Auto V,2013-09-17,False,https://media.rawg.io/media/games/20a/20aa03a1...,4.47,5,"[{'id': 5, 'title': 'exceptional', 'count': 44...",7414,...,0f0f0f,0f0f0f,"[{'platform': {'id': 187, 'name': 'PlayStation...","[{'platform': {'id': 1, 'name': 'PC', 'slug': ...","[{'id': 4, 'name': 'Action', 'slug': 'action',...","[{'id': 290376, 'store': {'id': 1, 'name': 'St...",None,"[{'id': 31, 'name': 'Singleplayer', 'slug': 's...","{'id': 4, 'name': 'Mature', 'slug': 'mature'}","[{'id': -1, 'image': 'https://media.rawg.io/me..."
1,3328,the-witcher-3-wild-hunt,The Witcher 3: Wild Hunt,2015-05-18,False,https://media.rawg.io/media/games/618/618c2031...,4.64,5,"[{'id': 5, 'title': 'exceptional', 'count': 56...",7238,...,0f0f0f,0f0f0f,"[{'platform': {'id': 187, 'name': 'PlayStation...","[{'platform': {'id': 1, 'name': 'PC', 'slug': ...","[{'id': 4, 'name': 'Action', 'slug': 'action',...","[{'id': 354780, 'store': {'id': 5, 'name': 'GO...",None,"[{'id': 31, 'name': 'Singleplayer', 'slug': 's...","{'id': 4, 'name': 'Mature', 'slug': 'mature'}","[{'id': -1, 'image': 'https://media.rawg.io/me..."
2,4200,portal-2,Portal 2,2011-04-18,False,https://media.rawg.io/media/games/2ba/2bac0e87...,4.58,5,"[{'id': 5, 'title': 'exceptional', 'count': 42...",6118,...,0f0f0f,0f0f0f,"[{'platform': {'id': 16, 'name': 'PlayStation ...","[{'platform': {'id': 1, 'name': 'PC', 'slug': ...","[{'id': 2, 'name': 'Shooter', 'slug': 'shooter...","[{'id': 465889, 'store': {'id': 2, 'name': 'Xb...",None,"[{'id': 31, 'name': 'Singleplayer', 'slug': 's...","{'id': 2, 'name': 'Everyone 10+', 'slug': 'eve...","[{'id': -1, 'image': 'https://media.rawg.io/me..."
3,4291,counter-strike-global-offensive,Counter-Strike: Global Offensive,2012-08-21,False,https://media.rawg.io/media/games/736/73619bd3...,3.57,4,"[{'id': 4, 'title': 'recommended', 'count': 17...",3639,...,0f0f0f,0f0f0f,"[{'platform': {'id': 4, 'name': 'PC', 'slug': ...","[{'platform': {'id': 1, 'name': 'PC', 'slug': ...","[{'id': 2, 'name': 'Shooter', 'slug': 'shooter...","[{'id': 4619, 'store': {'id': 3, 'name': 'Play...",None,"[{'id': 40847, 'name': 'Steam Achievements', '...","{'id': 4, 'name': 'Mature', 'slug': 'mature'}","[{'id': -1, 'image': 'https://media.rawg.io/me..."
4,5286,tomb-raider,Tomb Raider,2013-03-05,False,https://media.rawg.io/media/games/021/021c4e21...,4.06,4,"[{'id': 4, 'title': 'recommended', 'count': 24...",4104,...,0f0f0f,0f0f0f,"[{'platform': {'id': 16, 'name': 'PlayStation ...","[{'platform': {'id': 1, 'name': 'PC', 'slug': ...","[{'id': 4, 'name': 'Action', 'slug': 'action',...","[{'id': 33824, 'store': {'id': 7, 'name': 'Xbo...",None,"[{'id': 31, 'name': 'Singleplayer', 'slug': 's...","{'id': 4, 'name': 'Mature', 'slug': 'mature'}","[{'id': -1, 'image': 'https://media.rawg.io/me..."


### 2.2 Reference Data: Genres & Platforms

Genre and platform lookup data is small enough to fetch in a single call each (no pagination needed):

- `GET /genres` → genre reference table (`id`, `name`, `slug`, `games_count`)
- `GET /platforms` → platform reference table (`id`, `name`, `slug`, `games_count`, `year_start`)

These become the dimension tables in the warehouse.


In [38]:
response = get_data(endpoint="genres")
response = response.json()
df_genres = pd.DataFrame(response["results"])
df_genres.head()

,id,name,slug,games_count,image_background,games
0,4,Action,action,191810,https://media.rawg.io/media/games/d58/d588947d...,"[{'id': 3498, 'slug': 'grand-theft-auto-v', 'n..."
1,51,Indie,indie,86733,https://media.rawg.io/media/games/b54/b54598d1...,"[{'id': 1030, 'slug': 'limbo', 'name': 'Limbo'..."
2,3,Adventure,adventure,152135,https://media.rawg.io/media/games/879/879c930f...,"[{'id': 3439, 'slug': 'life-is-strange-episode..."
3,5,RPG,role-playing-games-rpg,62060,https://media.rawg.io/media/games/c6b/c6bfece1...,"[{'id': 3328, 'slug': 'the-witcher-3-wild-hunt..."
4,10,Strategy,strategy,62505,https://media.rawg.io/media/games/eeb/eeb9e668...,"[{'id': 10243, 'slug': 'company-of-heroes-2', ..."


In [39]:
response = get_data(endpoint="platforms")
response = response.json()
df_platforms = pd.DataFrame(response["results"])
df_platforms.head()

,id,name,slug,games_count,image_background,image,year_start,year_end,games
0,4,PC,pc,561287,https://media.rawg.io/media/games/d58/d588947d...,None,NaN,None,"[{'id': 3498, 'slug': 'grand-theft-auto-v', 'n..."
1,187,PlayStation 5,playstation5,1530,https://media.rawg.io/media/games/20a/20aa03a1...,None,2020.0,None,"[{'id': 3498, 'slug': 'grand-theft-auto-v', 'n..."
2,1,Xbox One,xbox-one,5744,https://media.rawg.io/media/games/618/618c2031...,None,NaN,None,"[{'id': 3498, 'slug': 'grand-theft-auto-v', 'n..."
3,18,PlayStation 4,playstation4,7042,https://media.rawg.io/media/games/34b/34b1f185...,None,NaN,None,"[{'id': 3498, 'slug': 'grand-theft-auto-v', 'n..."
4,186,Xbox Series S/X,xbox-series-x,1283,https://media.rawg.io/media/games/b29/b294fdd8...,None,2020.0,None,"[{'id': 3498, 'slug': 'grand-theft-auto-v', 'n..."


---
## 3. Transformation Layer

Raw API responses are denormalized JSON — nested lists for genres/platforms, mixed types, and redundant fields that aren't useful for analytics. The transformation step:

1. Drops fields that add no analytical value (internal RAWG metadata like `slug`, `saturated_color`, `dominant_color`, `stores`, `clip`, `tags`, etc.).
2. Casts `released`/`updated` to proper `datetime` types.
3. **Flattens the many-to-many relationships**: `genres` and `platforms` are exploded into their own long-format junction tables (`genres_raw`, `platforms_raw`) linking `game_id` to `genre_id`/`platform_id`, which is the correct normalized shape for a star schema (one game can have many genres and many platforms).
4. Extracts `esrb_rating.name` into a flat `esrb_rating` string column.
5. Adds derived analytics fields: `release_year`, `release_decade`, `rating_category` (Exceptional / Recommended / Meh / Skip), `metacritic_category` (Must Play / Great / Good / Mixed / N/A), `has_metacritic` flag, and a blended `popularity_score`.
6. Runs basic data-quality checks (deduplication by `game_id`, valid rating range, non-negative counts) before the data is loaded downstream.


In [40]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 29 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  200 non-null    int64  
 1   slug                200 non-null    object 
 2   name                200 non-null    object 
 3   released            200 non-null    object 
 4   tba                 200 non-null    bool   
 5   background_image    200 non-null    object 
 6   rating              200 non-null    float64
 7   rating_top          200 non-null    int64  
 8   ratings             200 non-null    object 
 9   ratings_count       200 non-null    int64  
 10  reviews_text_count  200 non-null    int64  
 11  added               200 non-null    int64  
 12  added_by_status     200 non-null    object 
 13  metacritic          195 non-null    float64
 14  playtime            200 non-null    int64  
 15  suggestions_count   200 non-null    int64  
 16  updated 

In [41]:
# 1. Drop columns that carry no analytical value
unused_columns = ["slug",
                  "reviews_text_count",
                  "added",
                  "added_by_status",
                  "suggestions_count",
                  "user_game",
                  "saturated_color",
                  "dominant_color",
                  "stores",
                  "clip",
                  "tags"]

df_raw.drop(columns=unused_columns, inplace=True)


In [42]:
# 2. Fix datetime dtypes
df_raw["released"] = pd.to_datetime(df_raw["released"], errors="coerce")
df_raw["updated"] = pd.to_datetime(df_raw["updated"], errors="coerce")


In [43]:
df_raw.head()

,id,name,released,tba,background_image,rating,rating_top,ratings,ratings_count,metacritic,playtime,updated,reviews_count,platforms,parent_platforms,genres,esrb_rating,short_screenshots
0,3498,Grand Theft Auto V,2013-09-17,False,https://media.rawg.io/media/games/20a/20aa03a1...,4.47,5,"[{'id': 5, 'title': 'exceptional', 'count': 44...",7414,92.0,74,2026-07-05 04:51:07,7549,"[{'platform': {'id': 187, 'name': 'PlayStation...","[{'platform': {'id': 1, 'name': 'PC', 'slug': ...","[{'id': 4, 'name': 'Action', 'slug': 'action',...","{'id': 4, 'name': 'Mature', 'slug': 'mature'}","[{'id': -1, 'image': 'https://media.rawg.io/me..."
1,3328,The Witcher 3: Wild Hunt,2015-05-18,False,https://media.rawg.io/media/games/618/618c2031...,4.64,5,"[{'id': 5, 'title': 'exceptional', 'count': 56...",7238,92.0,43,2026-07-05 04:49:39,7358,"[{'platform': {'id': 187, 'name': 'PlayStation...","[{'platform': {'id': 1, 'name': 'PC', 'slug': ...","[{'id': 4, 'name': 'Action', 'slug': 'action',...","{'id': 4, 'name': 'Mature', 'slug': 'mature'}","[{'id': -1, 'image': 'https://media.rawg.io/me..."
2,4200,Portal 2,2011-04-18,False,https://media.rawg.io/media/games/2ba/2bac0e87...,4.58,5,"[{'id': 5, 'title': 'exceptional', 'count': 42...",6118,95.0,11,2026-06-27 08:52:52,6175,"[{'platform': {'id': 16, 'name': 'PlayStation ...","[{'platform': {'id': 1, 'name': 'PC', 'slug': ...","[{'id': 2, 'name': 'Shooter', 'slug': 'shooter...","{'id': 2, 'name': 'Everyone 10+', 'slug': 'eve...","[{'id': -1, 'image': 'https://media.rawg.io/me..."
3,4291,Counter-Strike: Global Offensive,2012-08-21,False,https://media.rawg.io/media/games/736/73619bd3...,3.57,4,"[{'id': 4, 'title': 'recommended', 'count': 17...",3639,81.0,64,2026-07-03 05:12:50,3676,"[{'platform': {'id': 4, 'name': 'PC', 'slug': ...","[{'platform': {'id': 1, 'name': 'PC', 'slug': ...","[{'id': 2, 'name': 'Shooter', 'slug': 'shooter...","{'id': 4, 'name': 'Mature', 'slug': 'mature'}","[{'id': -1, 'image': 'https://media.rawg.io/me..."
4,5286,Tomb Raider,2013-03-05,False,https://media.rawg.io/media/games/021/021c4e21...,4.06,4,"[{'id': 4, 'title': 'recommended', 'count': 24...",4104,86.0,10,2026-07-02 18:25:47,4137,"[{'platform': {'id': 16, 'name': 'PlayStation ...","[{'platform': {'id': 1, 'name': 'PC', 'slug': ...","[{'id': 4, 'name': 'Action', 'slug': 'action',...","{'id': 4, 'name': 'Mature', 'slug': 'mature'}","[{'id': -1, 'image': 'https://media.rawg.io/me..."


In [44]:
# Flatten genres into a long-format game <-> genre junction table
genres_raw = df_raw[["id", "genres"]]
genres_raw = genres_raw.explode("genres", ignore_index=True)
genres_raw = genres_raw.rename({"id": "game_id"}, axis=1)

genre = pd.json_normalize(genres_raw["genres"])

genres_raw = pd.concat([genres_raw, genre], axis=1)
genres_raw = genres_raw.rename({"id": "genre_id"}, axis=1)
genres_raw = genres_raw.drop(columns=["genres", "slug"])

genres_raw


,game_id,genre_id,name,games_count,image_background
0,3498,4,Action,191809,https://media.rawg.io/media/games/49c/49c3dfa4...
1,3328,4,Action,191809,https://media.rawg.io/media/games/49c/49c3dfa4...
2,3328,5,RPG,62059,https://media.rawg.io/media/games/7c4/7c448374...
3,4200,2,Shooter,59643,https://media.rawg.io/media/games/d58/d588947d...
4,4200,7,Puzzle,97455,https://media.rawg.io/media/games/198/1988a337...
...,...,...,...,...,...
390,11973,4,Action,191809,https://media.rawg.io/media/games/49c/49c3dfa4...
391,11973,5,RPG,62059,https://media.rawg.io/media/games/7c4/7c448374...
392,23027,4,Action,191809,https://media.rawg.io/media/games/49c/49c3dfa4...
393,23027,3,Adventure,152132,https://media.rawg.io/media/games/b6b/b6b20bfc...


In [45]:
# Flatten platforms into a long-format game <-> platform junction table
platforms_raw = df_raw[["id", "platforms"]]
platforms_raw = platforms_raw.explode(["platforms"], ignore_index=True)
platforms_raw["platforms"] = [i["platform"] for i in platforms_raw["platforms"]]
platforms_raw = platforms_raw.rename({"id": "game_id"}, axis=1)

platform = pd.json_normalize(platforms_raw["platforms"])

platforms_raw = pd.concat([platforms_raw, platform], axis=1)
platforms_raw = platforms_raw.rename({"id": "platform_id"}, axis=1)
platforms_raw.drop(columns=["platforms",
                            "slug",
                            "image",
                            "year_end",
                            "year_start"], inplace=True)
platforms_raw.head()


,game_id,platform_id,name,games_count,image_background
0,3498,187,PlayStation 5,1528,https://media.rawg.io/media/games/152/152e788b...
1,3498,186,Xbox Series S/X,1283,https://media.rawg.io/media/games/b29/b294fdd8...
2,3498,16,PlayStation 3,3209,https://media.rawg.io/media/games/8e4/8e4de3f5...
3,3498,4,PC,561279,https://media.rawg.io/media/games/7cf/7cfc9220...
4,3498,18,PlayStation 4,7040,https://media.rawg.io/media/games/4be/4be6a6ad...


In [46]:
# Drop the now-redundant nested columns from the fact table
# (genre/platform detail now lives in genres_raw / platforms_raw instead)
df_raw.drop(columns=["ratings",
                     "platforms",
                     "parent_platforms",
                     "genres",
                     "short_screenshots"
                    ], inplace=True)


In [47]:
# Flatten esrb_rating.name into a single string column
df_raw["esrb_rating"] = [i["name"] if i is not None else None for i in df_raw['esrb_rating']]


In [48]:
# Checkpoint export — keep an intermediate copy of the cleaned tables on disk
df_raw.to_csv("raw_game.csv", index=False)
genres_raw.to_csv("genres_raw.csv", index=False)
platforms_raw.to_csv("platforms_raw.csv", index=False)


---
## 4. Load Layer: Star Schema Design for BigQuery

The warehouse follows a lightweight star schema so it's easy to query and to plug straight into a BI tool:

**Fact table** — `fact_games`: one row per game, holding all the numeric measures (rating, metacritic, playtime, reviews, etc.) plus the derived analytics fields from the transform step.

**Junction / dimension tables**:
- `genres_raw` — game ↔ genre junction (many-to-many)
- `platforms_raw` — game ↔ platform junction (many-to-many)
- `genre` — genre dimension (`genre_id`, `name`, `games_count`)
- `platform` — platform dimension (`platform_id`, `name`, `games_count`, `year_start`)

This shape keeps `fact_games` fast to scan for headline metrics while still supporting genre/platform-level breakdowns via joins — the same pattern a production analytics warehouse would use.


In [49]:
genres_raw.sort_values("game_id")

,game_id,genre_id,name,games_count,image_background
169,28,4,Action,191809,https://media.rawg.io/media/games/49c/49c3dfa4...
248,28,4,Action,191809,https://media.rawg.io/media/games/49c/49c3dfa4...
90,28,4,Action,191809,https://media.rawg.io/media/games/49c/49c3dfa4...
327,28,4,Action,191809,https://media.rawg.io/media/games/49c/49c3dfa4...
11,28,4,Action,191809,https://media.rawg.io/media/games/49c/49c3dfa4...
...,...,...,...,...,...
147,290856,2,Shooter,59643,https://media.rawg.io/media/games/d58/d588947d...
146,290856,4,Action,191809,https://media.rawg.io/media/games/49c/49c3dfa4...
67,290856,4,Action,191809,https://media.rawg.io/media/games/49c/49c3dfa4...
384,290856,2,Shooter,59643,https://media.rawg.io/media/games/d58/d588947d...


In [50]:
# Feature engineering: release year (null for titles that have not released yet)
df_raw["release_year"] = df_raw["released"].dt.year


In [51]:
# Feature engineering: bucket player rating into a readable category
def rating_category(rating):
    if rating >= 4.5: return "Exceptional"
    elif rating >= 3.5: return "Recommended"
    elif rating >= 2.5: return "Meh"
    else: return "Skip"

df_raw["rating_category"] = df_raw["rating"].apply(rating_category)


In [52]:
# Feature engineering: bucket critic score into a readable category
def metacritic_category(metacritic):
    if metacritic is None: return "N/A"
    elif metacritic >= 90: return "Must Play"
    elif metacritic >= 75: return "Great"
    elif metacritic >= 60: return "Good"
    else: return "Mixed"

df_raw["metacritic_category"] = df_raw["metacritic"].apply(metacritic_category)


In [53]:
# Additional feature engineering: release decade, metacritic availability flag,
# and a blended popularity score combining player rating and critic score
df_raw["release_decade"] = (df_raw["release_year"] // 10 * 10).astype("Int64").astype(str) + "s"
df_raw.loc[df_raw["release_year"].isna(), "release_decade"] = None

df_raw["has_metacritic"] = df_raw["metacritic"].notna()

df_raw["popularity_score"] = df_raw.apply(
    lambda r: (r["rating"] * 20) * 0.6 + (r["metacritic"] * 0.4)
    if pd.notna(r["metacritic"]) else r["rating"] * 20,
    axis=1
)

# Data-quality checks before loading downstream
df_raw = df_raw.drop_duplicates(subset="id")
assert df_raw["rating"].between(0, 5).all(), "rating out of expected 0-5 range"
assert (df_raw["ratings_count"] >= 0).all(), "ratings_count should never be negative"
assert (df_raw["playtime"] >= 0).all(), "playtime should never be negative"

print(f"Quality checks passed — {len(df_raw)} unique games ready to load.")


Quality checks passed — 40 unique games ready to load.


In [54]:
# Refresh the checkpoint file with the final enriched fact table
df_raw.to_csv("raw_game.csv", index=False)


In [55]:
# BigQuery configuration
PROJECT_ID = os.getenv("PROJECT_ID")
DATASET_ID = "RAWG"

# Service account credentials
CREDENTIAL_PATH = os.getenv("CREDENTIAL_PATH")
SCOPES = ["https://www.googleapis.com/auth/bigquery"]

credentials = service_account.Credentials.from_service_account_file(filename=CREDENTIAL_PATH, scopes=SCOPES)


In [56]:
# Load the fact table into BigQuery
TABLE_NAME = "fact_games"

pandas_gbq.to_gbq(
    dataframe=df_raw.drop_duplicates(),
    destination_table=f"{DATASET_ID}.{TABLE_NAME}",
    project_id=PROJECT_ID,
    credentials=credentials,
    if_exists="replace"
)


100%|██████████| 1/1 [00:00<00:00, 3371.63it/s]


In [57]:
df_genres.drop(columns="games", inplace=True)

In [58]:
df_platforms.drop(columns="games", inplace=True)

In [59]:
# Load the game <-> genre junction table
pandas_gbq.to_gbq(
    dataframe=genres_raw.drop_duplicates(),
    destination_table=f"{DATASET_ID}.genres_raw",
    project_id=PROJECT_ID,
    credentials=credentials,
    if_exists="replace"
)


100%|██████████| 1/1 [00:00<00:00, 6232.25it/s]


In [60]:
# Load the game <-> platform junction table
pandas_gbq.to_gbq(
    dataframe=platforms_raw.drop_duplicates(),
    destination_table=f"{DATASET_ID}.platforms_raw",
    project_id=PROJECT_ID,
    credentials=credentials,
    if_exists="replace"
)


100%|██████████| 1/1 [00:00<00:00, 10485.76it/s]


In [61]:
# Load the genre dimension table
pandas_gbq.to_gbq(
    dataframe=df_genres.drop_duplicates(),
    destination_table=f"{DATASET_ID}.genre",
    project_id=PROJECT_ID,
    credentials=credentials,
    if_exists="replace"
)


100%|██████████| 1/1 [00:00<00:00, 6875.91it/s]


In [62]:
# Load the platform dimension table
pandas_gbq.to_gbq(
    dataframe=df_platforms,
    destination_table=f"{DATASET_ID}.platform",
    project_id=PROJECT_ID,
    credentials=credentials,
    if_exists="replace"
)


100%|██████████| 1/1 [00:00<00:00, 9532.51it/s]


---
## 5. Analytics Layer: Dashboard-Ready SQL

With the star schema in BigQuery, the last step is a set of Custom SQL queries designed to plug directly into Looker Studio charts.

| Query | Looker Studio Chart | Purpose |
|---|---|---|
| 5.1 | KPI Scorecard Pack | Headline totals: game count, avg rating, avg metacritic, % with a critic score |
| 5.2 | Time series | Release trend per year, broken down by genre |
| 5.3 | Scatter plot | Player rating vs. Metacritic score, bubble size = playtime |
| 5.4 | Heatmap | Rating category × genre matrix |

### 5.1 KPI Scorecard Pack

```sql
SELECT
  COUNT(*)                                              AS total_games,
  ROUND(AVG(rating), 2)                                 AS avg_rating,
  ROUND(AVG(metacritic), 1)                             AS avg_metacritic,
  ROUND(100 * SUM(CAST(has_metacritic AS INT64)) / COUNT(*), 1) AS pct_with_metacritic
FROM `RAWG.fact_games`;
```

### 5.2 Release Trend per Year by Genre

```sql
SELECT
  f.release_year,
  g.name AS genre_name,
  COUNT(DISTINCT f.game_id) AS total_games
FROM `RAWG.fact_games` f
JOIN `RAWG.genres_raw` gr ON f.game_id = gr.game_id
JOIN `RAWG.genre` g       ON gr.genre_id = g.id
WHERE f.release_year IS NOT NULL
GROUP BY f.release_year, g.name
ORDER BY f.release_year;
```

### 5.3 Rating vs. Metacritic (bubble = playtime)

```sql
SELECT
  game_id,
  name,
  rating,
  metacritic,
  playtime
FROM `RAWG.fact_games`
WHERE has_metacritic = TRUE;
```

### 5.4 Rating Category × Genre Heatmap

```sql
SELECT
  g.name AS genre_name,
  f.rating_category,
  COUNT(DISTINCT f.game_id) AS total_games
FROM `RAWG.fact_games` f
JOIN `RAWG.genres_raw` gr ON f.game_id = gr.game_id
JOIN `RAWG.genre` g       ON gr.genre_id = g.id
GROUP BY g.name, f.rating_category;
```


---
## 6. Key Takeaways

- Built a repeatable ETL pipeline that turns a paginated third-party REST API into an analytics-ready warehouse in BigQuery, with no manual intervention beyond providing credentials.
- Modeled a many-to-many relationship (games ↔ genres, games ↔ platforms) as proper junction tables rather than flattening it away, which keeps genre/platform-level analysis accurate.
- Added a blended `popularity_score` metric and readable rating/metacritic buckets so business stakeholders can filter and segment without writing their own scoring logic.
- Included lightweight data-quality assertions (valid rating range, no negative counts, deduplication) as a guardrail before data reaches the warehouse.
- Wrote the SQL layer with Looker Studio consumption in mind, so the warehouse tables map directly onto scorecards, time series, scatter, and heatmap visuals.

### Possible Extensions

- Incremental/delta loads instead of full `if_exists="replace"` refreshes.
- Orchestration with Airflow/Cloud Composer for scheduled runs.
- A `dim_date` table and SCD handling for platform/genre reference data.
- Unit tests around the transform functions (`rating_category`, `metacritic_category`, `popularity_score`).
